# 1 · The Tokenizer

A language model reads numbers, not text, so the first thing to build is the converter between them. This notebook trains a 4,096-token byte-level BPE tokenizer on TinyStories and writes one file: `artifacts/tokenizer.json`. Everything downstream loads that file and never thinks about tokenization again.

Eight token IDs are spoken for before training starts. Token 0 separates stories. Tokens 1 and 2 are ChatML's turn markers — unused until post-training, but the vocabulary freezes into the model's weight shapes, so they have to exist now. The five blanks are insurance: a reserved token can be given a job later by renaming it in `tokenizer.json` and putting it in training data — no model surgery. (Llama 3.1's tool-calling tokens were Llama 3 reserved slots, renamed.)

In [4]:
from pathlib import Path

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPreTokenizer
from tokenizers.trainers import BpeTrainer

VOCAB_SIZE = 4096

SPECIAL_TOKENS = [
    "<|endoftext|>",  # token 0 — the story separator
    "<|im_start|>",   # token 1 — ChatML: a turn begins
    "<|im_end|>",     # token 2 — ChatML: a turn ends
    "<|reserved_3|>",
    "<|reserved_4|>",
    "<|reserved_5|>",
    "<|reserved_6|>",
    "<|reserved_7|>",
]

ARTIFACTS = Path("artifacts")

The corpus: 2.7 million synthetic children's stories, plain English. Look at one before doing anything to it.

In [5]:
dataset = load_dataset("karpathy/tinystories-gpt4-clean", split="train")

print(dataset)
print()
print(dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 2732634
})

Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.
Once the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!
The more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that.


BPE training is a census, then a loop. One pass through the corpus counts word frequencies; after that the corpus is never consulted again. Then, starting from single bytes: count every adjacent pair of symbols across the frequency table, merge the most frequent pair into a new token, record the rule, repeat. It runs exactly 3,832 times — 8 specials + 256 bytes + 3,832 merges = 4,096. Vocabulary size is a budget, not a convergence; the last merge is just where the music stopped.

Two settings matter here.

`add_prefix_space=False` — the pre-tokenizer attaches each space to the word that follows it. `False` means the encoding is exactly the text we gave it, nothing added. The alternative quietly prepends a space (so first words look like all the others), which fails round-trip and grows spaces nobody typed wherever text gets concatenated.

`initial_alphabet` — forces all 256 byte-symbols into the vocabulary whether the corpus uses them or not. TinyStories touches about 75 distinct characters; the rest are dead weight until a stranger pastes an emoji, which then degrades into raw bytes instead of being lost. This line is why a byte-level tokenizer needs no unknown-token, ever.

In [6]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevelPreTokenizer(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=ByteLevelPreTokenizer.alphabet(),
)


def batches_of_text(batch_size=1_000):
    for start in range(0, len(dataset), batch_size):
        yield dataset[start : start + batch_size]["text"]


tokenizer.train_from_iterator(batches_of_text(), trainer=trainer, length=len(dataset))

The artifact. The vocabulary is data; the ranked merge rules are the program; both live in this one file.

In [7]:
ARTIFACTS.mkdir(exist_ok=True)
tokenizer.save(str(ARTIFACTS / "tokenizer.json"))

The reserved block, in its assigned seats.

In [8]:
for token in SPECIAL_TOKENS:
    print(f"{tokenizer.token_to_id(token):4d}  {token}")

assert tokenizer.token_to_id("<|endoftext|>") == 0

   0  <|endoftext|>
   1  <|im_start|>
   2  <|im_end|>
   3  <|reserved_3|>
   4  <|reserved_4|>
   5  <|reserved_5|>
   6  <|reserved_6|>
   7  <|reserved_7|>


Encode-then-decode must reproduce the input exactly. A hundred documents sampled evenly across the corpus.

In [9]:
stride = len(dataset) // 100
for i in range(0, len(dataset), stride):
    text = dataset[i]["text"]
    ids = tokenizer.encode(text).ids
    assert tokenizer.decode(ids) == text

print("round trip: exact, 100/100 documents")

round trip: exact, 100/100 documents


`Ġ` is a space in costume: byte-level tokenizers remap every byte to a printable character, and space lands on `Ġ`. Mid-sentence words wear their leading space like a hat (`Ġtook`), while a text-opening word goes bare — `Lily` earned her own hatless token because thousands of stories start with her.

In [15]:
sentence = "Lily took her lantern to the zoo to see the brave dragon."

encoding = tokenizer.encode(sentence)
print(" | ".join(encoding.tokens))

Lily | Ġtook | Ġher | Ġl | an | tern | Ġto | Ġthe | Ġzoo | Ġto | Ġsee | Ġthe | Ġbrave | Ġdragon | .


One number for intuition: about four characters per token. All later context-length arithmetic rests on this ratio.

In [11]:
sample = dataset[:10_000]["text"]

total_characters = sum(len(text) for text in sample)
total_tokens = sum(len(tokenizer.encode(text).ids) for text in sample)

print(f"{total_characters / total_tokens:.2f} characters per token")

3.97 characters per token
